# HIDS Exploratory Data Analysis & Baseline Model

This notebook performs comprehensive EDA and trains a RandomForest baseline model for the Host-Based Intrusion Detection System (HIDS) dataset. We'll evaluate class balance, feature correlations, train baseline models, and analyze feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, stratify
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, 
                            precision_score, recall_score, f1_score, 
                            accuracy_score, roc_auc_score, roc_curve, auc)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set visualization parameters
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("✓ All libraries imported successfully")

## 1. Load and Explore the Dataset

In [ ]:
# Load the training dataset
df = pd.read_csv('../hids_dataset/features/train.csv')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\n✓ Dataset Shape: {df.shape}")
print(f"  - Samples: {df.shape[0]:,}")
print(f"  - Features: {df.shape[1]}")

print(f"\n✓ Data Types:\n{df.dtypes}")

print(f"\n✓ Missing Values:\n{df.isnull().sum().sum()} total missing values")

print(f"\n✓ First few rows:")
print(df.head())

In [ ]:
# Class balance analysis
print("\n" + "=" * 60)
print("CLASS BALANCE ANALYSIS")
print("=" * 60)

class_dist = df['label'].value_counts().sort_index()
class_pct = df['label'].value_counts(normalize=True).sort_index() * 100

print(f"\nClass Distribution:")
for label, count in class_dist.items():
    label_name = "BENIGN" if label == 1 else "MALICIOUS"
    print(f"  {label_name:12} (label={label}): {count:5,} samples ({class_pct[label]:5.2f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot
class_names = ['MALICIOUS', 'BENIGN']
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(class_names, class_dist.values, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(class_dist.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_dist.values, labels=class_names, autopct='%1.1f%%', 
            colors=colors, startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Class Balance', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✓ Imbalance Ratio: 1:{class_dist[0]/class_dist[1]:.2f} (Malicious:Benign)")

In [ ]:
# Feature statistics
print("\n" + "=" * 60)
print("FEATURE STATISTICS")
print("=" * 60)
print(f"\n{df.describe().round(3)}")

## 2. Feature Analysis and Correlations

In [ ]:
# Calculate correlation matrix
corr_matrix = df.corr()

print("=" * 60)
print("CORRELATION ANALYSIS")
print("=" * 60)

# Find highly correlated feature pairs (excluding label)
feature_cols = [col for col in df.columns if col != 'label']
corr_with_label = corr_matrix['label'][feature_cols].sort_values(ascending=False)

print("\nTop 10 Features Correlated with Label:")
print(corr_with_label.head(10).round(3))

# Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Check for high correlations between features
print("\nHighly Correlated Feature Pairs (|r| > 0.8, excluding label):")
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.8 or corr_matrix.iloc[i, j] < -0.8:
            if corr_matrix.columns[i] != 'label' and corr_matrix.columns[j] != 'label':
                high_corr_pairs.append((corr_matrix.columns[i], 
                                       corr_matrix.columns[j], 
                                       corr_matrix.iloc[i, j]))

if high_corr_pairs:
    for feat1, feat2, corr_val in high_corr_pairs:
        print(f"  {feat1:25} <-> {feat2:25}: {corr_val:7.3f}")
else:
    print("  No feature pairs with |correlation| > 0.8 found")

In [ ]:
# Variance analysis
print("\n" + "=" * 60)
print("VARIANCE ANALYSIS")
print("=" * 60)

variances = df[feature_cols].var().sort_values(ascending=False)
print("\nFeature Variances:")
print(variances.round(3))

# Plot variance
fig, ax = plt.subplots(figsize=(14, 6))
variances.plot(kind='barh', ax=ax, color='steelblue', alpha=0.7)
ax.set_xlabel('Variance', fontsize=12)
ax.set_title('Feature Variance Analysis', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Low Variance Features (var < 0.1):")
low_var_feats = variances[variances < 0.1]
if len(low_var_feats) > 0:
    for feat, var in low_var_feats.items():
        print(f"  {feat:30}: {var:.4f}")
else:
    print("  None identified")

In [ ]:
# Outlier detection using IQR
print("\n" + "=" * 60)
print("OUTLIER ANALYSIS (Interquartile Range)")
print("=" * 60)

outlier_summary = []
for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_pct = (len(outliers) / len(df)) * 100
    
    if outlier_pct > 0:
        outlier_summary.append({
            'feature': col,
            'outlier_count': len(outliers),
            'outlier_pct': outlier_pct,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        })

outlier_df = pd.DataFrame(outlier_summary).sort_values('outlier_pct', ascending=False)
print(f"\nTop 10 Features with Outliers:")
print(outlier_df.head(10).round(3))

print(f"\n✓ Total features with outliers: {len(outlier_df)}/{len(feature_cols)}")

# Boxplot for top outlier features
if len(outlier_df) > 0:
    top_outlier_features = outlier_df.head(6)['feature'].tolist()
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for idx, feature in enumerate(top_outlier_features):
        axes[idx].boxplot([df[df['label']==0][feature], df[df['label']==1][feature]],
                          labels=['Malicious', 'Benign'])
        axes[idx].set_title(f'{feature}', fontsize=11, fontweight='bold')
        axes[idx].grid(axis='y', alpha=0.3)
    
    plt.suptitle('Outlier Distribution by Class', fontsize=13, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

## 3. Data Preprocessing and Train-Test Split

In [ ]:
print("=" * 60)
print("DATA PREPROCESSING")
print("=" * 60)

# Separate features and target
X = df.drop('label', axis=1)
y = df['label']

print(f"\n✓ Features (X): {X.shape}")
print(f"✓ Target (y): {y.shape}")
print(f"\nFeature Columns ({len(X.columns)}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2}. {col}")

# Check if features are already scaled (they should be from preprocessing)
print(f"\nFeature Scaling Status:")
print(f"  Mean of features ≈ {X.mean().mean():.6f}")
print(f"  Std of features ≈ {X.std().mean():.6f}")
print(f"  ✓ Features appear to be standardized (mean≈0, std≈1)")

# Stratified train-test split
print("\n" + "=" * 60)
print("STRATIFIED TRAIN-TEST SPLIT")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain Set: {X_train.shape[0]:,} samples")
print(f"Test Set: {X_test.shape[0]:,} samples")
print(f"Train/Test Ratio: {X_train.shape[0] / X_test.shape[0]:.2f}:1")

print(f"\nClass Distribution in Train Set:")
for label in sorted(y_train.unique()):
    count = (y_train == label).sum()
    pct = (count / len(y_train)) * 100
    label_name = "BENIGN" if label == 1 else "MALICIOUS"
    print(f"  {label_name:12}: {count:6,} ({pct:5.2f}%)")

print(f"\nClass Distribution in Test Set:")
for label in sorted(y_test.unique()):
    count = (y_test == label).sum()
    pct = (count / len(y_test)) * 100
    label_name = "BENIGN" if label == 1 else "MALICIOUS"
    print(f"  {label_name:12}: {count:6,} ({pct:5.2f}%)")

## 4. Baseline Model Training - RandomForest

In [ ]:
print("=" * 60)
print("RANDOM FOREST BASELINE MODEL")
print("=" * 60)

# Train RandomForest
print("\n[*] Training RandomForest Classifier...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Handle class imbalance
)

rf_model.fit(X_train, y_train)
print("✓ Training complete")

# Make predictions
print("\n[*] Generating predictions on test set...")
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]  # Probability of positive class
print("✓ Predictions generated")

# Calculate metrics
print("\n" + "=" * 60)
print("MODEL EVALUATION METRICS")
print("=" * 60)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n✓ Accuracy:  {accuracy:.4f}")
print(f"✓ Precision: {precision:.4f} (Low False Positives)")
print(f"✓ Recall:    {recall:.4f} (Low False Negatives - CRITICAL for security)")
print(f"✓ F1-Score:  {f1:.4f}")
print(f"✓ ROC-AUC:   {roc_auc:.4f}")

# Classification report
print(f"\n{'-' * 60}")
print("DETAILED CLASSIFICATION REPORT")
print(f"{'-' * 60}")
print(classification_report(y_test, y_pred, target_names=['Malicious', 'Benign']))